In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
import json

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=groq_api_key
)

In [2]:
def get_video_id(url):
    if "v=" in url:
        video_id = url.split("v=")[1]
        video_id = video_id.split("&")[0]
        return video_id
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
        video_id = video_id.split("?")[0]
        return video_id
    else:
        return None

def get_transcript(url):
    video_id = get_video_id(url)
    if not video_id:
        print("Invalid Youtube URL")
        return None
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return transcript
    except Exception as e:
        print("Error: Could not fetch transcript. The video may not have captions or may not exist.")
        return None

def chunk_transcript(transcript, chunk_duration=120):
    chunks = []
    current_text = ""
    chunk_start = transcript[0].start

    for segment in transcript:
        if segment.start - chunk_start >= chunk_duration:
            chunks.append({
                "text": current_text.strip(),
                "start": chunk_start,
                "end": segment.start
            })
            current_text = segment.text + " "
            chunk_start = segment.start
        else:
            current_text += segment.text + " "

    # Don't forget the last chunk that's still being built
    if current_text:
        chunks.append({
            "text": current_text.strip(),
            "start": chunk_start,
            "end": transcript[-1].start
        })

    return chunks


class TranscriptionAgent:
    def __init__(self):
        self.name = "Transcription Agent"
    
    def run(self, url):
        transcript = get_transcript(url)
        if not transcript:
            return None
        chunks = chunk_transcript(transcript)
        return chunks

In [3]:
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent: {agent1.name}")
print(f"Chunks generated: {len(chunks)}")

Agent: Transcription Agent
Chunks generated: 10


In [4]:
class ConceptExtractionAgent:
    def __init__(self, llm):
        self.name = "Concept Extraction Agent"
        self.llm = llm
    
    def run(self, chunks):
        all_concepts = []
        
        for chunk in chunks:
            start_min = int(chunk["start"] // 60)
            start_sec = int(chunk["start"] % 60)
            end_min = int(chunk["end"] // 60)
            end_sec = int(chunk["end"] % 60)
            timestamp = f"{start_min}:{start_sec:02d} - {end_min}:{end_sec:02d}"
            
            prompt = f"""Extract the 2-3 most important concepts or topics from this transcript section.

Transcript:
{chunk['text']}

Return ONLY JSON in this format:
{{"concepts": ["concept 1", "concept 2", "concept 3"]}}
"""
            response = self.llm.invoke(prompt)
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
        
            concepts = json.loads(content)
            concepts["timestamp"] = timestamp
            all_concepts.append(concepts)
        
        return all_concepts

In [5]:
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)

for c in concepts:
    print(f"\n[{c['timestamp']}]")
    print(f"Concepts: {c['concepts']}")


[0:04 - 2:05]
Concepts: ['Neural Networks', 'Machine Learning', 'Image Recognition']

[2:05 - 4:06]
Concepts: ['Neural Networks', 'Neurons and Activations', 'Network Layers']

[4:06 - 6:07]
Concepts: ['Neural Network Structure', 'Digit Recognition', 'Layered Information Processing']

[6:07 - 8:08]
Concepts: ['Layered Neural Network Structure', 'Feature Detection and Recognition', 'Hierarchical Abstraction in Image Recognition']

[8:08 - 10:10]
Concepts: ['Neural Network Architecture', 'Weighted Sum', 'Feature Extraction']

[10:10 - 12:12]
Concepts: ['Sigmoid Function', 'Neural Network Weights and Biases', 'Activation of Neurons']

[12:12 - 14:14]
Concepts: ['Neural Network Architecture', 'Weights and Biases', 'Network Training and Learning']

[14:14 - 16:15]
Concepts: ['Linear Algebra', 'Neural Network Architecture', 'Matrix Vector Multiplication']

[16:15 - 18:15]
Concepts: ['Neural Network Training', 'Sigmoid Function', 'ReLU (Rectified Linear Unit)']

[18:15 - 18:25]
Concepts: ['Re

In [6]:
class QuestionGenerationAgent:
    def __init__(self, llm):
        self.name = "Question Generation Agent"
        self.llm = llm
    
    def run(self, concepts_list):
        all_questions = []
        
        for item in concepts_list:
            concepts = ", ".join(item["concepts"])
            timestamp = item["timestamp"]
            
            prompt = f"""Generate 1 multiple choice question that tests understanding of these concepts: {concepts}

The question should have:
- 4 options (A, B, C, D)
- The correct answer
- An explanation of why that answer is correct

Return ONLY JSON in this format:
{{
    "question": "the question text",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "correct_answer": "B",
    "explanation": "why this is correct"
}}
"""
            response = self.llm.invoke(prompt)
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
            
            question = json.loads(content)
            question["timestamp"] = timestamp
            question["concepts"] = item["concepts"]
            all_questions.append(question)
        
        return {"questions": all_questions}

In [7]:
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)

for q in quiz["questions"]:
    print(f"\n[{q['timestamp']}] Concepts: {q['concepts']}")
    print(f"Q: {q['question']}")


[0:04 - 2:05] Concepts: ['Neural Networks', 'Machine Learning', 'Image Recognition']
Q: What type of machine learning model is commonly used for image recognition tasks, such as classifying objects in images?

[2:05 - 4:06] Concepts: ['Neural Networks', 'Neurons and Activations', 'Network Layers']
Q: What is the primary purpose of the activation function in a neural network neuron?

[4:06 - 6:07] Concepts: ['Neural Network Structure', 'Digit Recognition', 'Layered Information Processing']
Q: In a neural network designed for digit recognition, which of the following best describes the role of the hidden layers in the network's structure?

[6:07 - 8:08] Concepts: ['Layered Neural Network Structure', 'Feature Detection and Recognition', 'Hierarchical Abstraction in Image Recognition']
Q: In a layered neural network structure for image recognition, what is the primary role of the early layers in the hierarchy?

[8:08 - 10:10] Concepts: ['Neural Network Architecture', 'Weighted Sum', 'Feat

In [10]:
class EvaluationAgent:
    def __init__(self, llm):
        self.name = "Evaluation Agent"
        self.llm = llm
    
    def run(self, quiz_data):
        score = 0
        weak_topics = []
        
        for i, q in enumerate(quiz_data["questions"]):
            print(f"\nQuestion {i+1}: {q['question']}")
            for option, text in q["options"].items():
                print(f"  {option}) {text}")
            user_answer = input("\nYour answer (A/B/C/D): ").upper()
            if user_answer == q['correct_answer']:
                score = score + 1
                print("Correct!")
            else:
                print("Incorrect!")
                weak_topics.extend(q['concepts'])
                weak_topics = list(set(weak_topics))
            print(f"Correct Answer: {q['correct_answer']}")
            print(f"Explanation: {q['explanation']}")
            print(f"Review in video at: {q['timestamp']}")
            
        print(f"\nFinal Score: {score}/{len(quiz_data['questions'])}")
        print(f"\nWeak Topics: {weak_topics}")

        return {
            "score": score,
            "total": len(quiz_data["questions"]),
            "weak_topics": weak_topics
        }

In [11]:
# Agent 1: Get transcript
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent 1 done: {len(chunks)} chunks")

# Agent 2: Extract concepts
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)
print(f"Agent 2 done: {len(concepts)} concept sets")

# Agent 3: Generate questions
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)
print(f"Agent 3 done: {len(quiz['questions'])} questions")

# Agent 4: Evaluate
agent4 = EvaluationAgent(llm)
results = agent4.run(quiz)

Agent 1 done: 10 chunks
Agent 2 done: 10 concept sets
Agent 3 done: 10 questions

Question 1: What type of machine learning model is commonly used for image recognition tasks, such as classifying objects in images?
  A) Linear Regression
  B) Convolutional Neural Networks
  C) Decision Trees
  D) Support Vector Machines



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: Convolutional Neural Networks (CNNs) are a type of neural network specifically designed for image recognition tasks. They use convolutional and pooling layers to extract features from images, allowing them to learn complex patterns and classify objects with high accuracy. This makes them the most suitable choice for image recognition tasks.
Review in video at: 0:04 - 2:05

Question 2: What is the primary function of the activation function in a neural network neuron?
  A) To reduce the dimensionality of the input data
  B) To introduce non-linearity into the neuron's output
  C) To increase the number of layers in the network
  D) To decrease the number of neurons in a layer



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: The activation function is used to introduce non-linearity into the neuron's output, allowing the neural network to learn and represent more complex relationships between inputs and outputs. Without an activation function, the neuron's output would be a linear combination of its inputs, limiting the network's ability to learn and generalize.
Review in video at: 2:05 - 4:06

Question 3: What is the primary function of the hidden layers in a neural network used for digit recognition and pattern analysis?
  A) To accept user input and display the output
  B) To apply non-linear transformations to the input data, allowing the network to learn complex patterns
  C) To store the network's weights and biases
  D) To perform the final classification or prediction


...

Incorrect!
Correct Answer: B
Explanation: Matrix-vector multiplication is used to transform input data into a higher-dimensional space, which is a fundamental operation in neural networks. This transformation allows the model to learn complex relationships between the input data and the output. In linear algebra, matrix-vector multiplication is used to apply linear transformations to vectors, which is essential for neural network functionality.
Review in video at: 14:14 - 16:15

Question 9: Which of the following activation functions is commonly used in the hidden layers of a neural network due to its ability to introduce non-linearity without saturating like the sigmoid function?
  A) Sigmoid function
  B) ReLU function
  C) Tanh function
  D) Linear function



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: The ReLU (Rectified Linear Unit) function is commonly used in the hidden layers of a neural network because it introduces non-linearity without saturating like the sigmoid function. ReLU maps all negative values to 0 and all positive values to the same value, which helps to avoid the vanishing gradient problem that can occur with sigmoid. The sigmoid function, on the other hand, can saturate, meaning that its output will be close to 0 or 1 for large input values, which can lead to vanishing gradients during backpropagation. The tanh function also saturates, although it has a range of -1 to 1 instead of 0 to 1 like sigmoid. The linear function does not introduce non-linearity, so it is not typically used as an activation function in hidden layers.
Review in video at: 16:15 - 18:15

Question 10: What is the primary purpose of the ReLU activation function in Deep Neural Networks?
  A) To introduce non-linearity and allow the network to learn compl


Your answer (A/B/C/D):  a


Correct!
Correct Answer: A
Explanation: ReLU, or Rectified Linear Unit, is a widely used activation function in Deep Neural Networks. Its primary purpose is to introduce non-linearity into the network, allowing it to learn complex relationships between inputs and outputs. However, one of the drawbacks of ReLU is that it can result in 'dying neurons', where neurons with negative inputs become stuck in a state where they do not contribute to the network's output. This is because ReLU outputs 0 for any negative input, effectively 'killing' the neuron.
Review in video at: 18:15 - 18:25

Final Score: 1/10

Weak Topics: ['Machine Learning', 'Weights and Biases', 'Sigmoid Function', 'Sigmoid and ReLU Functions', 'Neural Network Structure', 'Neural Network Training', 'Neural Network Layers', 'Linear Algebra', 'Activation Functions', 'Neural Networks', 'Neural Network Functionality', 'Neural Network Architecture', 'Activation and Information Processing', 'Network Learning', 'Feature Extraction'

In [16]:
def run_pipeline(url, num_questions=10):
    # Agent 1
    agent1 = TranscriptionAgent()
    chunks = agent1.run(url)
    if not chunks:
        return None
    print(f"Agent 1 done: {len(chunks)} chunks")
    
    # Select chunks evenly spread across the video
    total_chunks = len(chunks)
    if total_chunks <= num_questions:
        selected_chunks = chunks
    else:
        step = total_chunks // num_questions
        selected_chunks = [chunks[i * step] for i in range(num_questions)]
    print(f"Selected {len(selected_chunks)} chunks for quiz")
    
    # Agent 2: Extract concepts
    agent2 = ConceptExtractionAgent(llm)
    concepts = agent2.run(selected_chunks)
    if not concepts:
        return None
    else:
        print(f"Agent 2 done: {len(concepts)} concept sets")
     
    # Agent 3
    agent3 = QuestionGenerationAgent(llm)
    quiz = agent3.run(concepts)
    if not quiz:
        return None
    else:
        print(f"Agent 3 done: {len(quiz['questions'])} questions")
    
    # Agent 4
    agent4 = EvaluationAgent(llm)
    results = agent4.run(quiz)
    
    # Return results
    return results

In [17]:
results = run_pipeline("https://www.youtube.com/watch?v=_gGRBWaRpAQ&list=PLZoTAELRMXVNNrHSKv36Lr3_156yCo6Nn&index=5")

Agent 1 done: 37 chunks
Selected 10 chunks for quiz
Agent 2 done: 10 concept sets
Agent 3 done: 10 questions

Question 1: In an NLP class, a student is participating in a quiz competition and is asked to generate text based on a given prompt. If the student uses a pre-trained model to generate the text without properly citing the source, what would be the most appropriate description of this action?
  A) A demonstration of excellent NLP skills
  B) A violation of academic integrity
  C) A necessary step to complete the task efficiently
  D) A common practice in NLP competitions



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: B
Explanation: This is correct because using someone else's work, such as a pre-trained model, without proper citation or credit is a form of plagiarism, which is a violation of academic integrity. In academic settings, it is essential to properly cite sources and give credit where it is due to maintain the integrity of one's work.
Review in video at: 0:14 - 2:14

Question 2: Which of the following is a primary purpose of removing stop words during text preprocessing in a Bag of Words model?
  A) To reduce the dimensionality of the feature space by removing rare words
  B) To eliminate common words like 'the' and 'and' that do not add much value to the meaning of the text
  C) To increase the weight of nouns and verbs in the text
  D) To convert all text to lowercase



Your answer (A/B/C/D):  b


Correct!
Correct Answer: B
Explanation: Stop words are common words like 'the', 'and', 'a', etc. that do not add much value to the meaning of the text. Removing them helps reduce the noise in the data and improves the performance of the model by focusing on more meaningful words. This is a key step in text preprocessing for Bag of Words models.
Review in video at: 6:21 - 8:23

Question 3: What does cosine similarity measure between two vectors in a vector representation of semantic meaning?
  A) The dot product of the vectors
  B) The cosine of the angle between the vectors, which represents their semantic similarity
  C) The magnitude of the difference between the vectors
  D) The Euclidean distance between the vectors


...

Correct!
Correct Answer: B
Explanation: This is correct because high-order N-grams result in a large number of unique sequences, many of which may appear only once or a few times in the training data. This leads to a sparsity problem, where there is not enough data to reliably estimate the probabilities of these sequences, making it difficult to train accurate models.
Review in video at: 42:47 - 44:48

Question 9: What technique is used to reduce the dimensionality of a vector in feature extraction, where the importance of each word in a document is determined by its term frequency?
  A) Term Frequency-Inverse Document Frequency (TF-IDF) alone
  B) Combining Term Frequency with Vector Reduction techniques such as Principal Component Analysis (PCA) or Singular Value Decomposition (SVD)
  C) Only Principal Component Analysis (PCA)
  D) Only Singular Value Decomposition (SVD)



Your answer (A/B/C/D):  d


Incorrect!
Correct Answer: B
Explanation: This is correct because while Term Frequency-Inverse Document Frequency (TF-IDF) helps in understanding the importance of each word in a document, techniques like Principal Component Analysis (PCA) or Singular Value Decomposition (SVD) are used for reducing the dimensionality of the vector. Thus, combining Term Frequency with vector reduction techniques is essential for effective feature extraction and vector reduction.
Review in video at: 48:51 - 50:53

Question 10: What is the primary purpose of calculating document frequency in a spam classifier?
  A) To determine the importance of each word in the entire corpus
  B) To remove common words like 'the' and 'and' that do not add much value to the classification
  C) To calculate the term frequency of each word in a document
  D) To perform text preprocessing techniques like tokenization and stemming



Your answer (A/B/C/D):  c


Incorrect!
Correct Answer: B
Explanation: Calculating document frequency helps in identifying and potentially removing common words (stopwords) that appear in many documents, as they do not add much value to the classification of spam vs non-spam emails. This step is crucial in text preprocessing for a spam classifier.
Review in video at: 54:55 - 56:55

Final Score: 5/10

Weak Topics: ['logistic regression with log base e', 'Feature Extraction', 'term frequency importance', 'Vector Reduction', 'Term Frequency Calculation', 'Text Preprocessing', 'Inverse Document Frequency', 'Sentence Analysis', 'Quiz Competition', 'Document Frequency', 'vector multiplication', 'Term Frequency', 'Academic Integrity', 'Spam Classifier', 'NLP Class']
